# Edge Plasma profiles

This notebook demonstrates how to load and visualize edge plasma profiles using the `cherab.imas` interface.
Here, we propose how to visualize edge plasmas with grid meshes defined in the IMAS data structure.

The example test data was calculated by SOLPS-ITER for an ITER 15 MA H-mode scenario.

In [ ]:
import numpy as np
import ultraplot as uplt
from imas import DBEntry
from matplotlib.colors import SymLogNorm
from rich import print as rprint

from cherab.imas.datasets import iter_jintrac, iter_solps
from cherab.imas.ggd import GGDGrid
from cherab.imas.ids.common import get_ids_time_slice
from cherab.imas.ids.common.ggd import load_grid
from cherab.imas.ids.edge_profiles import load_edge_species

# Set dark background for plots
uplt.rc.style = "dark_background"

## Define a function to plot edge plasma profiles

In [ ]:
def plot_grid_quantity(
    ax: uplt.axes.Axes,
    grid: GGDGrid,
    quantity: np.ndarray,
    title: str = "",
    title_center: str = "",
    clabel: str = "",
    logscale: bool = False,
    symmetric: bool = False,
    cbar_kwargs: dict = None,
) -> uplt.axes.Axes:
    """Plot a quantity defined on a grid."""
    ax = grid.plot_mesh(data=quantity, ax=ax)

    if logscale:
        # Plot lowest values (mainly 0's) on linear map, as log(0) = -inf.
        linthresh = np.percentile(np.unique(quantity), 1)
        norm = SymLogNorm(
            linthresh=float(max(linthresh, 1.0e-10 * quantity.max())),
            base=10,
        )
        ax.collections[0].set_norm(norm)

    if symmetric:
        vmax = np.abs(quantity.max())
        ax.collections[0].set_clim(-vmax, vmax)
        ax.collections[0].set_cmap("berlin")
    else:
        ax.collections[0].set_cmap("gnuplot")

    ax.colorbar(
        ax.collections[0],
        formatter="log" if logscale else None,
        tickminor=True,
        **(cbar_kwargs or {}),
    )

    if title_center:
        ax.text(
            0.5,
            0.55,
            title_center,
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=14,
        )

    ax.format(
        aspect="equal",
        xlabel="$R$ [m]",
        ylabel="$Z$ [m]",
        xlocator=1,
        ylocator=1,
        title=title,
    )
    return ax

## Retrieve the sample data

In [ ]:
path = iter_solps()

## Load grid and species data

### Plot all grid subsets

In [edge_profiles IDS](https://imas-data-dictionary.readthedocs.io/en/latest/generated/ids/edge_profiles.html), there are multiple grid subsets defined.
Here, we see what grid subsets are available and plot them all.

In [ ]:
# Load edge_profiles IDs
with DBEntry(path, "r") as entry:
    ids = get_ids_time_slice(
        entry,
        "edge_profiles",
        time=0,
    )

# Load grid object
grid, subsets, subset_id = load_grid(
    ids.grid_ggd[0],
    with_subsets=True,
)

# Print available grid subsets
rprint("Available grid subsets:", subset_id)

In [ ]:
subset_groups = [
    [
        "Cells",
    ],
    [
        "Inner core",
        "Outer core",
        "Inner SOL",
        "Outer SOL",
        "Lower inner divertor",
        "Lower outer divertor",
    ],
    [
        "CORE",
        "SOL",
        "OUTER_DIVERTOR",
        "INNER_DIVERTOR",
        "Inner Midplane",
        "Outer Midplane",
        "Neutral pressure cells",
    ],
]

fig, axs = uplt.subplots(ncols=len(subset_groups))

for ax, subset_names in zip(axs, subset_groups, strict=True):
    for i, subset_name in enumerate(subset_names):
        grid_subset = grid.subset(subsets[subset_name])
        grid_subset.plot_mesh(ax=ax, label=subset_name, edgecolor=f"C{i}")

    ax.legend(ncols=1, loc="center")

axs.format(
    xlim=(4.0, 8.5),
    ylim=(-4.7, 4.8),
    grid=True,
    xlocator=1,
    ylocator=1,
    tickminor=True,
)

### Load edge species data

We choose the `"Cells"` subset covering the entire edge region and load the corresponding edge species data. The `edge_profiles` IDS contains multiple species, and we can choose which one to visualize.

In [ ]:
grid_cells = grid.subset(subsets["Cells"])

composition = load_edge_species(
    ids.ggd[0],
    grid_subset_index=subset_id["Cells"],
    split_ion_bundles=False,
)

rprint(composition)

## Plot edge plasma profiles

### Electron profiles

In [ ]:
# Electron density
fig, ax = uplt.subplots()
ax = plot_grid_quantity(
    ax,
    grid_cells,
    composition.electron.density,
    title_center="Electron density\n$n_\\mathrm{e}$ [m$^{-3}$]",
    logscale=True,
)
ax.format(
    xlim=(4.0, 8.5),
    ylim=(-4.7, 4.8),
    xlocator=1,
    ylocator=1,
    tickminor=True,
)

# Electron temperature
fig, ax = uplt.subplots()
ax = plot_grid_quantity(
    ax,
    grid_cells,
    composition.electron.temperature,
    title_center="Electron temperature\n$T_\\mathrm{e}$ [eV]",
    logscale=True,
)
ax.format(
    xlim=(4.0, 8.5),
    ylim=(-4.7, 4.8),
    xlocator=1,
    ylocator=1,
    tickminor=True,
)

### Species profiles

In [ ]:
# Store all data to plot in a list
data: list[tuple[np.ndarray, dict]] = []

for profile in composition.ion + composition.neutral + composition.molecule:
    charge = profile.species.z_min
    if (element := profile.species.element) is not None:
        symbol = element.symbol
    elif profile.species.elements:
        symbol = "-".join(element.symbol for element in profile.species.elements)
    else:
        symbol = "Unknown"

    if charge == 0:
        name = symbol
    elif charge == 1:
        name = f"{symbol}$^+$"
    else:
        name = f"{symbol}$^{{{charge}+}}$"

    # Density
    data.append(
        (
            profile.density,
            dict(
                title_center=f"{name} density [m$^{{-3}}$]",
                logscale=True,
            ),
        )
    )
    if (element := profile.species.element) is not None and element.atomic_number == 1:
        # Temperature
        if profile.temperature is not None and np.any(profile.temperature):
            data.append(
                (
                    profile.temperature,
                    dict(
                        title_center=f"{name} temperature [eV]",
                        logscale=True,
                    ),
                )
            )
        if charge:
            # Velocity profiles
            vpar = profile.velocity.parallel
            if vpar is not None and np.any(vpar):
                data.append(
                    (
                        vpar,
                        dict(
                            title_center=f"{name} parallel velocity [m/s]",
                            symmetric=True,
                        ),
                    )
                )
        else:
            for vtype in {"radial", "poloidal", "phi"}:
                velocity = getattr(profile.velocity, vtype)
                if velocity is not None and np.any(velocity):
                    data.append(
                        (
                            velocity,
                            dict(
                                title_center=f"{name} {vtype} velocity [m/s]",
                                symmetric=True,
                            ),
                        )
                    )

# Plot all data
fig, axes = uplt.subplots(
    ncols=3,
    nrows=int(np.ceil(len(data) / 3)),
)

for i_ax, (quantity, kwargs) in enumerate(data):
    ax = plot_grid_quantity(
        axes[i_ax],
        grid_cells,
        quantity,
        **kwargs,
        cbar_kwargs=dict(
            loc="lr",
            orientation="vertical",
            ticklabelsize="small",
            length=5,
            frame=False,
        ),
    )

axes.format(
    xlabel="",
    ylabel="",
    xtickloc="neither",
    ytickloc="neither",
    linestyle="none",
    grid=False,
)

## Split the bundled species profiles

### Retrieve the bundled species profiles

In [ ]:
path = iter_jintrac()

# Load edge_profiles IDs
with DBEntry(path, "r") as entry:
    ids = get_ids_time_slice(
        entry,
        "edge_profiles",
        time=0,
    )

# Load grid object
grid, subsets, subset_id = load_grid(
    ids.grid_ggd[0],
    with_subsets=True,
)

# Print available grid subsets
rprint("Available grid subsets:", subset_id)

grid_cells = grid.subset(subsets["cells"])

composition = load_edge_species(
    ids.ggd[0],
    grid_subset_index=subset_id["cells"],
    split_ion_bundles=False,
)

rprint(composition)

In [ ]:
from cherab.core.atomic.elements import neon
from cherab.imas.ids.common import solve_coronal_equilibrium

# Select one neon ion bundle
bundle = composition.ion_bundle[2]

# Solve coronal equilibrium
densities = solve_coronal_equilibrium(
    neon,
    bundle.density,
    composition.electron.density,
    composition.electron.temperature,
    z_min=bundle.species.z_min,
    z_max=bundle.species.z_max,
)

# Plot the split charge states
fig, axes = uplt.subplots(
    ncols=3,
    nrows=int(np.ceil(densities.shape[0] / 3)),
)

for i_ax, charge in enumerate(
    np.arange(bundle.species.z_min, bundle.species.z_max + 1, dtype=int),
):
    ax = plot_grid_quantity(
        axes[i_ax],
        grid_cells,
        densities[i_ax, :],
        title_center=f"{neon.symbol}$^{{{charge}+}}$ density [m$^{{-3}}$]",
        logscale=True,
        cbar_kwargs=dict(
            loc="lr",
            orientation="vertical",
            ticklabelsize="small",
            length=5,
            frame=False,
        ),
    )

axes.format(
    xlabel="",
    ylabel="",
    xtickloc="neither",
    ytickloc="neither",
    linestyle="none",
)